# From-Scratch Neural Network for Detecting Protected Information

Goal: Build a simple feedforward NN using only NumPy to classify text snippets as containing:
- PII (Personally Identifiable Information)
- HIPAA-related (health info patterns)
- PCI (payment card info)
- Other protected
- None

Approach:
- Manual forward + backpropagation
- No PyTorch/TensorFlow/autograd
- Synthetic or small public data
- Focus: correctness of math & gradients

Sections to follow:
1. Data preparation
2. Model architecture
3. Forward pass
4. Loss
5. Backward pass
6. Optimizer
7. Training loop
8. Evaluation

Step: I. Load libraries and plant random seed

In [ ]:
import numpy as np # only math engine
from sklearn.model_selection import train_test_split # quick & reliable data splitting
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer  # only for simple vectorization easiest way to turn text → fixed-size vectors (bag-of-words) without writing your own tokenizer from zero
import re  # for regex-based feature help later
import matplotlib.pyplot as plt  # you'll need this eventually for loss plots
import pandas as pd  # data manipulation and exporting to csv/excel
from datasets import load_dataset # for importing HuggingFace datasets
from google.colab import userdata # for using HuggingFace token
userdata.get('HF_TOKEN')
import json #for saving jsonl files
from pathlib import Path
import os
from google.colab import drive
drive.mount('/content/drive')

# Optional: set random seed for reproducibility
randseed = np.random.seed(42)

print("NumPy version:", np.__version__)
print("Setup complete – ready to build data and model.")
print("NP random seed: ", np.random.seed(42))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NumPy version: 2.0.2
Setup complete – ready to build data and model.
NP random seed:  None


Step II. Chang working directory to current Google Drive folder to access data files

In [ ]:
%cd "drive/MyDrive/Colab Notebooks/MIS 769/Project"

/content/drive/MyDrive/Colab Notebooks/MIS 769/Project


Step 1: Load cycle 1 data

In [ ]:
NON_PII_PATH    = "non_pii_fixed.jsonl"
PII_PART1_PATH  = "pii_partition_1.jsonl"

Check that data exists and path is correct

In [ ]:
files = [
    "non_pii_fixed.jsonl",
    "pii_partition_1.jsonl",
    "pii_partition_2.jsonl",
    "pii_partition_3.jsonl",
    "pii_partition_4.jsonl",
    "pii_partition_5.jsonl"
]

for f in files:
    path = f"{f}"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✓ {f} exists  ({size_mb:.1f} MB)")
    else:
        print(f"✗ {f} NOT found")

✓ non_pii_fixed.jsonl exists  (7.7 MB)
✓ pii_partition_1.jsonl exists  (12.0 MB)
✓ pii_partition_2.jsonl exists  (11.9 MB)
✓ pii_partition_3.jsonl exists  (11.9 MB)
✓ pii_partition_4.jsonl exists  (11.9 MB)
✓ pii_partition_5.jsonl exists  (11.9 MB)


Convert and concatenate the non-PII and PII-1 partitions to a pandas dataframe and display summary metrics


In [ ]:
# Define loading function (handles empty lines & minor JSON issues)
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Warning: Skipping invalid JSON at line {line_num} in {path}: {e}")
    return pd.DataFrame(data)

# Paths (files are already in /content)
NON_PII_PATH    = "non_pii_fixed.jsonl"
PII_PART1_PATH  = "pii_partition_1.jsonl"

print("Loading non-PII file...")
df_non = load_jsonl(NON_PII_PATH)

print("Loading PII partition 1...")
df_pii = load_jsonl(PII_PART1_PATH)

# Quick sanity checks
print("\nNon-PII shape:", df_non.shape)
print("PII part 1 shape:", df_pii.shape)

print("\nNon-PII columns:", list(df_non.columns))
print("PII part 1 columns:", list(df_pii.columns))

# Combine & shuffle
df_cycle1 = pd.concat([df_non, df_pii], ignore_index=True)
df_cycle1 = df_cycle1.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nCycle 1 total rows:", df_cycle1.shape[0])
print("Expected ≈ 6964 rows")

# Sample privacy_mask to understand the format
print("\nFirst 3 privacy_mask values (from shuffled df):")
print(df_cycle1['privacy_mask'].head(3).tolist())

print("\nLast 3 privacy_mask values:")
print(df_cycle1['privacy_mask'].tail(3).tolist())

# Quick count of empty vs non-empty masks (rough label check)
def is_empty_mask(m):
    if not isinstance(m, list):
        return True  # treat non-list as no PII
    return len(m) == 0

mask_status = df_cycle1['privacy_mask'].apply(is_empty_mask)
print("\nRough label distribution in cycle 1:")
print(mask_status.value_counts(normalize=True).round(3))
print("(True = appears to be no-PII / empty mask)")

Loading non-PII file...
Loading PII partition 1...

Non-PII shape: (3482, 9)
PII part 1 shape: (3482, 9)

Non-PII columns: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']
PII part 1 columns: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']

Cycle 1 total rows: 6964
Expected ≈ 6964 rows

First 3 privacy_mask values (from shuffled df):
[[], [{'value': 'F', 'start': 40, 'end': 41, 'label': 'SEX'}, {'value': '02', 'start': 43, 'end': 45, 'label': 'TIME'}, {'value': 'Sushama', 'start': 47, 'end': 54, 'label': 'GIVENNAME1'}, {'value': '02:29', 'start': 169, 'end': 174, 'label': 'TIME'}, {'value': '2029-10-06T00:00:00', 'start': 178, 'end': 197, 'label': 'DATE'}, {'value': 'Masculine', 'start': 234, 'end': 243, 'label': 'SEX'}, {'value': '4 PM', 'start': 245, 'end': 249, 'label': 'TIME'}, {'value': 'Haxhi', 'start': 251, 'end': 256, 'la

In [ ]:
display(df_cycle1.head())

,source_text,target_text,privacy_mask,span_labels,mbert_text_tokens,mbert_bio_labels,id,language,set
0,ticipants to engage in meaningful photo and vi...,ticipants to engage in meaningful photo and vi...,[],[],"[ti, ##ci, ##pan, ##ts, to, engage, in, meanin...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",45773C,English,train
1,"Reality Therapy Client Consent Form\n\nI, F, 0...","Reality Therapy Client Consent Form\n\nI, [SEX...","[{'value': 'F', 'start': 40, 'end': 41, 'label...","[[365, 371, ""GIVENNAME1""], [353, 363, ""TIME""],...","[Reality, Therapy, Cl, ##ient, Con, ##sent, Fo...","[O, O, O, O, O, O, O, O, O, O, O, B-SEX, B-TIM...",52598A,English,train
2,859\n- IP: 4713:73da:1ae:cd0e:8448:4e65:3578:1...,859\n- IP: [IP]\n- Country: [COUNTRY]\n- Postc...,[{'value': '4713:73da:1ae:cd0e:8448:4e65:3578:...,"[[121, 125, ""POSTCODE""], [75, 79, ""POSTCODE""],...","[859, -, IP, :, 471, ##3, :, 73, ##da, :, 1a, ...","[O, O, O, O, O, B-IP, I-IP, I-IP, I-IP, I-IP, ...",43616D,English,train
3,**Mentorship Program Impact Assessment Report*...,**Mentorship Program Impact Assessment Report*...,[],[],"[*, *, Men, ##tors, ##hip, Program, Impact, As...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",40899A,English,train
4,Please find below an example of Telecom Billin...,Please find below an example of Telecom Billin...,"[{'value': 'RG4{>jK0,Q', 'start': 232, 'end': ...","[[232, 242, ""PASS""]]","[Please, find, below, an, example, of, Telecom...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",51101A,English,train


Step 2: Create binary labels

In [ ]:
# Function to create binary label from privacy_mask (based on your data format)
def has_pii(mask):
    """True if privacy_mask has any PII entries"""
    if isinstance(mask, list):
        return len(mask) > 0
    return False  # Non-list? Treat as no PII (safety)

# Add the label column
df_cycle1['label'] = df_cycle1['privacy_mask'].apply(has_pii).astype(int)

# Extract for training: X_raw (texts), y (labels)
X_raw = df_cycle1['source_text'].values  # NumPy array of strings
y     = df_cycle1['label'].values        # NumPy array of 0s/1s

print("✅ Binary labels created!")
print(f"   y shape: {y.shape}  (6964 rows)")
print(f"   Class balance:\n"
      f"     No PII (0): {np.sum(y == 0):,d} rows ({(y == 0).mean():.1%})\n"
      f"     Has PII (1): {np.sum(y == 1):,d} rows ({(y == 1).mean():.1%})")

# Quick peek at 5 random examples
print("\nSample texts + labels:")
for i in np.random.choice(len(y), 5, replace=False):
    text_snip = X_raw[i][:120] + "..." if len(X_raw[i]) > 120 else X_raw[i]
    print(f"  [{y[i]}] {text_snip}")


✅ Binary labels created!
   y shape: (6964,)  (6964 rows)
   Class balance:
     No PII (0): 3,482 rows (50.0%)
     Has PII (1): 3,482 rows (50.0%)

Sample texts + labels:
  [1] **Psychological Resilience Intervention Proposal**
*Proposal for implementing interventions to enhance psychological res...
  [0] umulating $5 in fines, users will be suspended from using the facilities.</Suspension>
                </Overdue_Items>
...
  [1] ;+T'
      Timing: '7 PM'
      Last_Name1: 'Adesanya'
      Last_Name2: 'Von Zobeltitz'
    - Individual_ID: '2'
      ...
  [0] Subject: Collaboration Feedback for Early Intervention Program Development Initiative

Dear Team,

I hope this message f...
  [1] 905:** This license serves as proof of the holder's ability to operate a vehicle.

3. **0305146818:** This identificatio...


### Step 3: Text Vectorization (TF-IDF)

Now that we have:
- `X_raw`: array of raw text strings (source_text)
- `y`: binary labels (0 = no PII, 1 = has PII)

We convert the raw text into numerical vectors using **TF-IDF** (Term Frequency × Inverse Document Frequency).

This creates a fixed-length feature vector for each text example:
- Each dimension = one word or bigram from the vocabulary
- Value = TF-IDF score (importance of that term in this text)
- Most values = 0 (sparse representation)
- Vocabulary capped at ~20,000 most informative terms

Why TF-IDF? It emphasizes rare, distinctive patterns (great for PII like dates, emails, IDs) while down-weighting common words.

We use scikit-learn's TfidfVectorizer for this preprocessing step only — the model itself will be 100% from-scratch NumPy.

In [ ]:
# Create vectorizer with settings suited for PII detection
vectorizer = TfidfVectorizer(
    max_features=4000,       # max number of features (words + bigrams)
    ngram_range=(1, 2),       # unigrams + bigrams (helps catch phrases like dates, names)
    stop_words='english',     # remove common filler words
    min_df=2,                 # ignore terms in < 2 documents
    max_df=0.95,              # ignore terms in >95% of documents
    lowercase=True,           # normalize case
    norm='l2'                 # ← Explicitly add L2 normalization (helps a lot)
)

print("Fitting & transforming...")
X = vectorizer.fit_transform(X_raw).toarray().astype(np.float32)

# Results
print(f"\nX shape: {X.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")

# Quick inspection
print("\nTop 10 most distinctive terms (highest IDF):")
feature_names = vectorizer.get_feature_names_out()
print(feature_names[np.argsort(-vectorizer.idf_)[:10]])

print("\nSample vector (first row, first 8 non-zero features):")
nonzero = np.where(X[0] > 0)[0]
for idx in nonzero[:8]:
    print(f"  {feature_names[idx]:<20} : {X[0, idx]:.4f}")

Fitting & transforming...

X shape: (6964, 4000)
Vocabulary size: 4,000

Top 10 most distinctive terms (highest IDF):
['educator _______________________' 'patient email' 'com access' 'learner'
 'person person' 'person sex' 'user email' '________' 'com participant'
 'participant passport']

Sample vector (first row, first 8 non-zero features):
  adhere               : 0.1899
  agreement            : 0.1398
  behavior             : 0.1666
  community            : 0.1203
  connection           : 0.2217
  content              : 0.1500
  engage               : 0.1526
  environment          : 0.1304


In [ ]:
print(f"X shape after vectorizer: {X.shape}")
print(f"Sparsity: {(X == 0).mean():.1%} of values are zero")

X shape after vectorizer: (6964, 4000)
Sparsity: 99.3% of values are zero


### Step 4: ScratchNet – From-Scratch Binary Classifier

We now build a simple feed-forward neural network **entirely with NumPy** (no autograd, no PyTorch/TensorFlow).

Architecture for binary classification:
- Input: 20,000 features (our TF-IDF vectors)
- Hidden layers: e.g., 512 → 128 neurons (ReLU activation)
- Output: 1 neuron + sigmoid → probability of "has PII" (1)

We'll implement:
- Forward pass
- Binary cross-entropy loss
- Backward pass (manual gradients)
- Parameter updates (vanilla SGD)

No fancy optimizers yet — we keep it pure and educational.

In [ ]:
class ScratchNet:
    def __init__(self, input_size=20000, hidden_sizes=[512, 128], dropout=0.3):
        self.layers = []
        prev_size = input_size

        # Hidden layers
        for h in hidden_sizes:
            W = np.random.randn(prev_size, h) * np.sqrt(2.0 / prev_size)  # He initialization
            b = np.zeros(h)
            self.layers.append({'W': W, 'b': b})
            prev_size = h

        # Output layer (single neuron for binary)
        W_out = np.random.randn(prev_size, 1) * np.sqrt(2.0 / prev_size)
        b_out = np.zeros(1)
        self.layers.append({'W': W_out, 'b': b_out})

        self.dropout = dropout
        self.training = True  # toggle for dropout during train vs predict

    def forward(self, X):
        activations = [X]  # input is first activation
        # NEW: store dropout masks (only when applied)
        dropout_masks = []

        for i, layer in enumerate(self.layers[:-1]):
            Z = activations[-1] @ layer['W'] + layer['b']
            A = np.maximum(0, Z)  # ReLU
            if self.training and self.dropout > 0:
                mask = (np.random.rand(*A.shape) > self.dropout) / (1 - self.dropout)
                A *= mask  # inverted dropout (scale survivors)
            activations.append(A)

        # Output layer (logits)
        Z_out = activations[-1] @ self.layers[-1]['W'] + self.layers[-1]['b']
        activations.append(Z_out)

        # NEW: save for backward use
        self.activations = activations
        self.dropout_masks = dropout_masks

        return activations

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # avoid overflow

    def predict_proba(self, X):
        self.training = False               # turn off dropout (no randomness at inference)
        acts = self.forward(X)              # run full forward pass → get activations list
        logits = acts[-1]                   # the last element = Z_out (raw logits)
        proba = 1 / (1 + np.exp(-logits))   # or self.sigmoid(logits) — same thing
        self.training = True                # turn training mode back on (good practice)
        return proba.ravel()                # flatten to 1D array (batch_size,)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def backward(self, logits, y):
        """
        Backpropagation: compute gradients and store dW/db in each layer.
        Uses the actual activations storage: [X, A1, A2, ..., An, Z_out]
        """
        batch_size = y.shape[0]
        y = y.reshape(-1, 1)
        logits = logits.reshape(-1, 1)

        activations = self.activations
        dropout_masks = self.dropout_masks

        # Output delta
        sigmoid_out = 1 / (1 + np.exp(-logits))
        delta = sigmoid_out - y
        current_delta = delta

        # Loop backwards over layers (output layer → first hidden)
        for layer_idx in range(len(self.layers) - 1, -1, -1):
            layer = self.layers[layer_idx]

            # Determine previous activation index
            if layer_idx == len(self.layers) - 1:
                # Output layer: previous is last hidden A → index -2
                A_prev = activations[-2]
            else:
                # Hidden layer layer_idx: previous is activations[layer_idx]
                # (0 → X, 1 → A1, 2 → A2, ...)
                A_prev = activations[layer_idx]

            # Gradients
            dW = A_prev.T @ current_delta / batch_size
            db = np.mean(current_delta, axis=0)
            layer['dW'] = dW
            layer['db'] = db

            # Propagate delta to previous layer (skip if first layer)
            if layer_idx > 0:
                current_delta = current_delta @ layer['W'].T

                # ReLU derivative from A_prev (>0 where gradient flows)
                relu_deriv = (A_prev > 0).astype(float)
                current_delta *= relu_deriv

                # Dropout scaling for the previous layer
                mask_idx = layer_idx - 1  # masks are indexed 0 = first hidden, ...
                if mask_idx >= 0 and mask_idx < len(dropout_masks):
                    mask = dropout_masks[mask_idx]
                    if mask is not None:
                        current_delta *= mask / (1 - self.dropout)

        # End: all layers now have dW/db

    def update_parameters(self, lr: float):
        """Apply vanilla SGD updates using the gradients stored in each layer."""
        for layer in self.layers:
            layer['W'] -= lr * layer['dW']
            layer['b'] -= lr * layer['db']

**Step 5: Numerically Stable Binary Cross-Entropy Loss**

With the forward pass implemented in `ScratchNet`, we now add the loss function that measures how far the model's raw output logits are from the true binary labels (0 = no PII, 1 = has PII).

We implement **binary cross-entropy (BCE)** in a **numerically stable** way, working directly on the raw logits (`Z_out`) instead of probabilities. This avoids NaN/inf issues that occur when logits become very large during early training.

Key properties:
- Input: raw logits (pre-sigmoid) + binary labels
- Computation: avoids explicit sigmoid and log(near-zero)
- Output: single scalar (average loss over the batch)
- Stable even for extreme logit values (|z| > 50)

The chosen formulation is mathematically equivalent to standard BCE but safe:

loss = mean[ max(z, 0) − z · y + log(1 + exp(−|z|)) ]

This loss will be used both to monitor training progress and as the starting point for backpropagation (gradient w.r.t. logits = sigmoid(z) − y).

(Implementation of the loss function goes here.)

In [ ]:
def stable_bce_loss(logits: np.ndarray, y: np.ndarray) -> float:
    """
    Numerically stable binary cross-entropy loss.

    Args:
        logits: Raw model outputs (pre-sigmoid), shape (batch_size,) or (N,)
        y:      Binary labels {0, 1}, same shape as logits

    Returns:
        Scalar mean loss (positive float)
    """
    # Ensure correct shapes and types
    logits = np.asarray(logits, dtype=np.float64).ravel()
    y      = np.asarray(y,     dtype=np.float64).ravel()

    if logits.shape != y.shape:
        raise ValueError(f"Shape mismatch: logits {logits.shape}, y {y.shape}")

    # Stable BCE computation
    # Equivalent to: -mean[ y*log(sigmoid(z)) + (1-y)*log(1-sigmoid(z)) ]
    # but computed without ever evaluating sigmoid(z) explicitly
    loss_terms = (
        np.maximum(logits, 0.0)          # max(z, 0)
        - logits * y                     # -z * y
        + np.log1p(np.exp(-np.abs(logits)))  # log(1 + exp(-|z|))
    )

    return np.mean(loss_terms)

In [ ]:
# Quick sanity check
np.random.seed(42)
logits_test = np.array([-5.0, -2.0, 0.0, 2.0, 5.0])
y_test      = np.array([0,    0,    0,   1,   1])

print("Loss:", stable_bce_loss(logits_test, y_test))
# Should be around ~0.993 (very close to log(2) ≈ 0.693 for random + well-separated examples)

Loss: 0.19208677992482528


**Step 6: Training Loop – One Epoch on Cycle 1 Data**

With the forward pass, backward pass, parameter updates, and stable BCE loss now implemented in `ScratchNet` and as a standalone function, we now create the training loop that ties everything together.

We implement a simple loop that:
- Instantiates the model
- Uses the balanced cycle-1 dataset (`X` from TF-IDF, `y` binary labels)
- Runs one or more epochs of full-batch training
- Computes average loss and basic accuracy per epoch
- Prints progress for monitoring

This is the first real training execution: forward → loss → backward → parameter update.

(Implementation of the training loop goes here.)

In [ ]:
# Step 6: One-Epoch Training Loop (full-batch for starters)

from tqdm import tqdm  # optional – remove if you prefer no progress bar

# Instantiate the model
model = ScratchNet(input_size=20000, hidden_sizes=[512, 128], dropout=0.3)

# Use the already prepared cycle-1 data from previous cells
X_train = X         # TF-IDF features (6964, 20000)
y_train = y         # binary labels (6964,)

# Hyperparameters
lr = 0.001          # small starting value
epochs = 5          # start with 5; increase later

print("Starting training...")

for epoch in range(epochs):
    model.training = True  # enable dropout

    # Forward pass
    activations = model.forward(X_train)
    logits = activations[-1].ravel()  # Z_out as 1D array

    # Compute loss
    loss = stable_bce_loss(logits, y_train)

    # Backward pass
    model.backward(logits.reshape(-1, 1), y_train.reshape(-1, 1))

    # Parameter update
    model.update_parameters(lr=lr)

    # Accuracy for monitoring
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    acc = np.mean(preds == y_train)

    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss:.4f} | Acc: {acc:.4f}")

Starting training...
Epoch 1/5 | Loss: 0.6932 | Acc: 0.4941
Epoch 2/5 | Loss: 0.6932 | Acc: 0.4933
Epoch 3/5 | Loss: 0.6932 | Acc: 0.5007
Epoch 4/5 | Loss: 0.6932 | Acc: 0.4984
Epoch 5/5 | Loss: 0.6932 | Acc: 0.4983


**PII Detection from-Scratch NN Project – Quick Resume Summary (March 9, 2026)**

**Goal**  
Binary classifier (has PII vs no PII) built entirely from scratch in NumPy (manual forward/backprop, no autograd).  
Dataset: English subset of ai4privacy/pii-masking-300k (~29.9k rows, ~88% has-PII).  
Plan: train on 5 cycles of balanced 1:1 data (~7k per cycle), later extend to multi-class.

**Current Status – What is working**
- Data prep done (cycle 1):
  - `X` = TF-IDF (20000 features, dense np.float32 array, shape ~6964×20000)
  - `y` = binary labels (0/1 np array, shape 6964)
- `ScratchNet` class fully built:
  - He init, ReLU hidden layers, optional dropout (inverted)
  - forward: stores `self.activations = [X, A1, A2, ..., An, Z_out]` and `self.dropout_masks`
  - backward: computes & stores `dW`, `db` per layer (ReLU deriv from A>0, dropout scaling)
  - update_parameters: vanilla SGD (`W -= lr * dW`, `b -= lr * db`)
  - predict_proba / predict (dropout off at inference)
- `stable_bce_loss(logits, y)` implemented (numerically stable, logit form)
- Single-epoch full-batch training loop running (Step 6):
  - Instantiates model, uses full cycle-1 `X`/`y`
  - Forward → loss → backward → update → prints train loss & acc

**Current Problem / Last Results**
- Training loop runs without crashing, but model not learning:
  Epoch 1–5: Loss stuck at 0.6932, Accuracy ~49–50%
  → Logits near 0 everywhere → gradients likely vanishing or zero
- Last attempted fixes: (not yet tested in new session)
  - Try much higher lr (0.05–0.5)
  - Disable dropout (`dropout=0.0`)
  - Print gradient norms after backward to check if dW/db are non-zero
  - Print logits mean/std to confirm variance

**Immediate Next Steps (pick one when resuming)**
1. Add gradient norm printing in loop → see if gradients exist at all
2. Rerun with lr=0.1 and dropout=0.0 → check if loss moves
3. Add 85/15 train/val split before loop → monitor generalization
4. Reduce hidden sizes (e.g. [128]) → easier signal flow
5. Debug backward chain (if grads zero): check delta propagation layer-by-layer

**Notebook cell order reminder**
- Step 2: binary labels (`y`)
- Step 3: TF-IDF vectorization (`X`)
- Step 4: ScratchNet class (updated with backward + update_parameters)
- Step 5: stable_bce_loss function
- Step 6: training loop (full-batch on cycle 1)

Paste this summary + say “let’s debug the stuck loss” or “try higher lr” or whatever you want to tackle first.
Good night! Sleep well and crush work tomorrow.